# Notebook 1 – Data loading and inspection

Loading the raw YouControlMarket panel dataset (2018-2024) and performing an initial inspection of its structure and coverage.

**Steps performed:**
1. Loading raw CSV and casting column types
2. Displaying dataset structure and sample rows
3. Inspecting observation count per firm
4. Identifying firms with more than 7 observations

**Output:** raw `df` passed to Notebook 2 for cleaning.


> **Data confidentiality notice**
> The source dataset is proprietary and cannot be included in this repository.
> To illustrate the expected data format and column structure, the first 10 rows of the loaded DataFrame are displayed below.
> All firm identifiers in the sample output have been anonymised.

In [1]:
import pandas as pd
import numpy as np

## 1. Loading data

Reading the raw CSV file. All identifier/categorical columns keeping their original types; every other column is coerced to `float64`.

In [2]:
NON_NUMERIC_COLS = ['Firm ID', 'Firm Status', 'KVED', 'Sector',
                    'Date Registration', 'Region']

df = pd.read_csv(
    '2Final_YCM_ANONIMIZED_fin_data_2018-2024_BIG.csv',
    sep=';',
    encoding='utf-8',
    na_values=['NULL', 'null', 'NaN', '', '#VALUE!'],
    low_memory=False,
    dtype={'KVED': str}
)

# Coerce all numeric columns to float64
for col in df.columns:
    if col not in NON_NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)

print('Dataset shape:', df.shape)
print('\nData types:')
print(df.dtypes.value_counts())
print('\nNon-numeric columns:')
print(df[NON_NUMERIC_COLS].dtypes)
print('\nKVED sample values:')
print(df['KVED'].dropna().unique()[:10])

Dataset shape: (84644, 167)

Data types:
float64    161
object       5
int64        1
Name: count, dtype: int64

Non-numeric columns:
Firm ID               int64
Firm Status          object
KVED                 object
Sector               object
Date Registration    object
Region               object
dtype: object

KVED sample values:
['20.14' '01.46' '02.40' '02.10' '10.11' '86.10' '86.21' '49.39' '23.61'
 '68.20']


## 2. Dataset structure — first 10 rows

Displaying the first 10 rows to illustrate the column structure and data format.
The full dataset is confidential and not included in the repository.

In [3]:
df.head(10)

,Firm ID,Firm Status,Year,KVED,Sector,Date Registration,Firm Age,Region,Number of Employees,Intangible Assets,...,Employee Benefits Expense (Wages and Salaries),Social Security Contributions,Depreciation,Other Operating Expenses (by nature),Total Operating Expenses,Weighted Average Number of Ordinary Shares,Diluted Weighted Average Number of Ordinary Shares,Basic Earnings (Loss) per Share,Diluted Earnings (Loss) per Share,Dividends per Ordinary Share
0,1,Припинено,2018.0,20.14,C,08.12.2003,16.0,Чернівецька область,0.0,0.0,...,154.0,26.0,105.0,456.0,748.0,0.0,0.0,0.0,0.0,0.0
1,1,Припинено,2019.0,20.14,C,08.12.2003,17.0,Чернівецька область,0.0,0.0,...,195.0,35.0,105.0,228.0,571.0,0.0,0.0,0.0,0.0,0.0
2,1,Припинено,2021.0,20.14,C,08.12.2003,19.0,Чернівецька область,0.0,0.0,...,410.0,72.0,90.0,947.0,1529.0,0.0,0.0,0.0,0.0,0.0
3,1,Припинено,2022.0,20.14,C,08.12.2003,20.0,Чернівецька область,0.0,0.0,...,405.0,87.0,87.0,1874.0,2456.0,0.0,0.0,0.0,0.0,0.0
4,1,Припинено,2023.0,20.14,C,08.12.2003,21.0,Чернівецька область,0.0,0.0,...,419.0,92.0,34.0,3882.0,4432.0,0.0,0.0,0.0,0.0,0.0
5,2,Не перебуває в процесі припинення,2023.0,01.46,A,06.09.1994,30.0,Львівська область,214.0,0.0,...,15643.0,3472.0,25568.0,65079.0,415358.0,0.0,0.0,0.0,0.0,0.0
6,2,Не перебуває в процесі припинення,2024.0,01.46,A,06.09.1994,31.0,Львівська область,214.0,0.0,...,39686.0,8707.0,37187.0,142440.0,1123592.0,0.0,0.0,0.0,0.0,0.0
7,3,В стані припинення,2020.0,02.40,A,29.03.1999,22.0,Харківська область,0.0,26.0,...,23982.0,5315.0,2403.0,2337.0,59121.0,0.0,0.0,0.0,0.0,0.0
8,3,В стані припинення,2021.0,02.40,A,29.03.1999,23.0,Харківська область,0.0,3.0,...,19529.0,4242.0,2319.0,7833.0,62279.0,0.0,0.0,0.0,0.0,0.0
9,3,В стані припинення,2022.0,02.40,A,29.03.1999,24.0,Харківська область,0.0,0.0,...,22574.0,4763.0,2239.0,7334.0,63495.0,0.0,0.0,0.0,0.0,0.0


## 3. Observations per firm

Inspecting how many yearly observations each firm has. The panel covers 2018–2024, so the maximum expected count is 7.

In [4]:
def obs_count_per_firm(dataframe):
    """Return a DataFrame with observation counts per firm."""
    counts = dataframe.groupby('Firm ID')['Year'].count().reset_index()
    counts.columns = ['Firm ID', 'Num Years']
    return counts

obs_per_firm = obs_count_per_firm(df)

print('Distribution of observations per firm:')
print(
    obs_per_firm['Num Years']
    .value_counts()
    .sort_index()
    .rename('Firms')
    .to_frame()
    .assign(Share=lambda x: (x['Firms'] / x['Firms'].sum() * 100).round(1))
    .to_string()
)
print(f"\nMean years per firm  : {obs_per_firm['Num Years'].mean():.1f}")
print(f"Median years per firm: {obs_per_firm['Num Years'].median():.0f}")

Distribution of observations per firm:
           Firms  Share
Num Years              
1              1    0.0
2           3607   19.2
3           2949   15.7
4           2859   15.3
5           2428   13.0
6           3261   17.4
7           3632   19.4
8              2    0.0

Mean years per firm  : 4.5
Median years per firm: 4


## 4. Investigating firms with more than 7 observations

Listing firms that exceed the 2018–2024 window. Any such firm is outside the expected range and likely contains duplicate records.

In [5]:
firms_over_7 = obs_per_firm[obs_per_firm['Num Years'] > 7]['Firm ID'].values

print('Firms with more than 7 observations:')
print(
    df[df['Firm ID'].isin(firms_over_7)][['Firm ID', 'Year', 'Firm Status']]
    .sort_values(['Firm ID', 'Year'])
    .to_string(index=False)
)

Firms with more than 7 observations:
 Firm ID   Year                       Firm Status
    2546 2018.0 Не перебуває в процесі припинення
    2546 2019.0 Не перебуває в процесі припинення
    2546 2020.0 Не перебуває в процесі припинення
    2546 2020.0 Не перебуває в процесі припинення
    2546 2021.0 Не перебуває в процесі припинення
    2546 2022.0 Не перебуває в процесі припинення
    2546 2023.0 Не перебуває в процесі припинення
    2546 2024.0 Не перебуває в процесі припинення
    6254 2018.0 Не перебуває в процесі припинення
    6254 2019.0 Не перебуває в процесі припинення
    6254 2020.0 Не перебуває в процесі припинення
    6254 2020.0 Не перебуває в процесі припинення
    6254 2021.0 Не перебуває в процесі припинення
    6254 2022.0 Не перебуває в процесі припинення
    6254 2023.0 Не перебуває в процесі припинення
    6254 2024.0 Не перебуває в процесі припинення


In [6]:
# Identify duplicates before removal
dup_mask = df.duplicated(subset=['Firm ID', 'Year'], keep='first')
duplicates_df = df[dup_mask][['Firm ID', 'Year', 'Firm Status']].sort_values(['Firm ID', 'Year'])

print(f'Total duplicate observations: {len(duplicates_df):,}')
print(f"Unique firms affected       : {duplicates_df['Firm ID'].nunique():,}")
print('\nDuplicate observations:')
print(duplicates_df.to_string(index=False))

Total duplicate observations: 226
Unique firms affected       : 226

Duplicate observations:
 Firm ID   Year                       Firm Status
     428 2020.0 Не перебуває в процесі припинення
    1334 2020.0 Не перебуває в процесі припинення
    2076 2020.0 Не перебуває в процесі припинення
    2168 2020.0 Не перебуває в процесі припинення
    2546 2020.0 Не перебуває в процесі припинення
    2786 2020.0 Не перебуває в процесі припинення
    3032 2020.0 Не перебуває в процесі припинення
    3091 2020.0 Не перебуває в процесі припинення
    3378 2020.0 Не перебуває в процесі припинення
    3438 2020.0 Не перебуває в процесі припинення
    3848 2020.0 Не перебуває в процесі припинення
    4021 2020.0 Не перебуває в процесі припинення
    4087 2020.0 Не перебуває в процесі припинення
    4262 2020.0                         Припинено
    5387 2020.0 Не перебуває в процесі припинення
    6124 2020.0 Не перебуває в процесі припинення
    6254 2020.0 Не перебуває в процесі припинення
    658

In [7]:
# Remove duplicates
initial_obs = len(df)
df = df.drop_duplicates(subset=['Firm ID', 'Year'], keep='first').reset_index(drop=True)

print(f'Observations removed  : {initial_obs - len(df):,}')
print(f'Observations remaining: {len(df):,}')

# Verify: no firm should exceed 7 observations
obs_per_firm = obs_count_per_firm(df)
firms_over_7 = obs_per_firm[obs_per_firm['Num Years'] > 7]

if len(firms_over_7) == 0:
    print('\n No firms with more than 7 observations')
else:
    print(f"\n⚠️  Still {len(firms_over_7)} firms with more than 7 observations:")
    print(
        df[df['Firm ID'].isin(firms_over_7['Firm ID'])][['Firm ID', 'Year', 'Firm Status']]
        .sort_values(['Firm ID', 'Year'])
        .to_string(index=False)
    )

Observations removed  : 226
Observations remaining: 84,418

 No firms with more than 7 observations


## 5. Missing values analysis

Summarising missingness across the full dataset, then focusing on the key financial columns used in indicator construction. Dropping observations missing any of these columns.

In [8]:
# Dataset-wide summary
missing = (
    pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing %':     (df.isnull().sum() / len(df) * 100).round(2)
    })
    .sort_values('Missing %', ascending=False)
)
missing = missing[missing['Missing Count'] > 0]

print(f'Total columns with missing values: {len(missing)} out of {df.shape[1]}')
print(f'Total missing values: {df.isnull().sum().sum():,}')

Total columns with missing values: 42 out of 167
Total missing values: 10,300


In [9]:
# Columns required by financial indicator formulas (Number of Employees excluded)
KEY_COLS = [
    'Total Assets', 'Total Current Assets', 'Total Non-Current Assets',
    'Total Equity', 'Total Liabilities', 'Total Current Liabilities',
    'Total Long-term Liabilities', 'Total Debt',
    'Short-Term Bank Loans', 'Long-Term Bank Loans',
    'Net Revenue', 'COGS', 'Gross Profit',
    'EBIT', 'EBITDA', 'Net Income',
    'Financial Expenses', 'Depreciation',
    'PPE', 'Inventories', 'Accounts Receivable',
    'Accounts Payable', 'Cash and Cash Equivalents',
    'Net Debt', 'Firm Age',
]

print('Missing values in key financial columns:')
print('=' * 55)
for col in KEY_COLS:
    if col in df.columns:
        n_miss = df[col].isna().sum()
        pct    = n_miss / len(df) * 100
        status = 'ok ' if n_miss == 0 else 'not ok '
        print(f'  {status} {col:<40} {n_miss:>6,} ({pct:.1f}%)')

# Drop observations missing any key column
initial_obs = len(df)
df = df.dropna(subset=KEY_COLS).reset_index(drop=True)

print(f'\nObservations removed  : {initial_obs - len(df):,}')
print(f'Observations remaining: {len(df):,}')
print(f"Unique firms remaining: {df['Firm ID'].nunique():,}")

Missing values in key financial columns:
  not ok  Total Assets                                  2 (0.0%)
  not ok  Total Current Assets                          2 (0.0%)
  ok  Total Non-Current Assets                      0 (0.0%)
  not ok  Total Equity                                  1 (0.0%)
  not ok  Total Liabilities                             3 (0.0%)
  not ok  Total Current Liabilities                     3 (0.0%)
  ok  Total Long-term Liabilities                   0 (0.0%)
  ok  Total Debt                                    0 (0.0%)
  ok  Short-Term Bank Loans                         0 (0.0%)
  ok  Long-Term Bank Loans                          0 (0.0%)
  not ok  Net Revenue                                  51 (0.1%)
  not ok  COGS                                         50 (0.1%)
  not ok  Gross Profit                                  5 (0.0%)
  not ok  EBIT                                         12 (0.0%)
  not ok  EBITDA                                       12 (0.0%)
  no

## 6. Removing invalid negative values

Checking balance-sheet and income-statement items that must be non-negative by accounting definition. Removing all observations that violate these constraints.

In [10]:
NON_NEGATIVE_COLS = {
    # Assets
    'Total Assets': 'Total Assets cannot be negative',
    'Total Current Assets': 'Current Assets cannot be negative',
    'Total Non-Current Assets': 'Non-Current Assets cannot be negative',
    'PPE': 'PPE cannot be negative',
    'Inventories': 'Inventories cannot be negative',
    'Cash and Cash Equivalents': 'Cash cannot be negative',
    'Accounts Receivable': 'Accounts Receivable cannot be negative',
    # Liabilities
    'Total Current Liabilities': 'Current Liabilities cannot be negative',
    'Total Long-term Liabilities': 'Long-term Liabilities cannot be negative',
    'Total Liabilities': 'Total Liabilities cannot be negative',
    'Accounts Payable': 'Accounts Payable cannot be negative',
    'Total Debt': 'Total Debt cannot be negative',
    'Short-Term Bank Loans': 'Short-term loans cannot be negative',
    'Long-Term Bank Loans': 'Long-term loans cannot be negative',
    # Income statement
    'Net Revenue': 'Net Revenue cannot be negative',
    'COGS': 'COGS cannot be negative',
    'Depreciation': 'Depreciation cannot be negative',
    'Financial Expenses': 'Financial Expenses cannot be negative',
}

print('Negative values check:')

invalid_idx = set()
for col, reason in NON_NEGATIVE_COLS.items():
    if col not in df.columns:
        print(f' Column not found: {col}')
        continue

    mask  = df[col].notna() & (df[col] < 0)
    n_neg = mask.sum()

    if n_neg > 0:
        print(f"\n  not ok {col}")
        print(f"     Negative observations : {n_neg:,}")
        print(f"     Unique firms affected  : {df[mask]['Firm ID'].nunique():,}")
        print(f"     Reason                 : {reason}")
        invalid_idx.update(df[mask].index.tolist())
    else:
        print(f'  ok {col}')

print(f'Total observations with at least one negative value: {len(invalid_idx):,}')
print(f'Share of dataset: {len(invalid_idx) / len(df) * 100:.1f}%')

Negative values check:
  ok Total Assets
  ok Total Current Assets
  ok Total Non-Current Assets

  not ok PPE
     Negative observations : 7
     Unique firms affected  : 5
     Reason                 : PPE cannot be negative
  ok Inventories

  not ok Cash and Cash Equivalents
     Negative observations : 38
     Unique firms affected  : 31
     Reason                 : Cash cannot be negative

  not ok Accounts Receivable
     Negative observations : 16
     Unique firms affected  : 11
     Reason                 : Accounts Receivable cannot be negative

  not ok Total Current Liabilities
     Negative observations : 4
     Unique firms affected  : 4
     Reason                 : Current Liabilities cannot be negative

  not ok Total Long-term Liabilities
     Negative observations : 22
     Unique firms affected  : 15
     Reason                 : Long-term Liabilities cannot be negative

  not ok Total Liabilities
     Negative observations : 4
     Unique firms affected  : 4
    

In [11]:
# Remove observations with invalid negative values
initial_obs = len(df)

negative_mask = pd.Series(False, index=df.index)
for col in NON_NEGATIVE_COLS:
    if col in df.columns:
        negative_mask |= df[col].notna() & (df[col] < 0)

df = df[~negative_mask].reset_index(drop=True)

print(f'Observations removed  : {initial_obs - len(df):,}')
print(f'Observations remaining: {len(df):,}')
print(f"Unique firms remaining: {df['Firm ID'].nunique():,}")

Observations removed  : 173
Observations remaining: 84,182
Unique firms remaining: 18,721


## 7. Balance sheet integrity checks

Verifying eleven accounting identities and inequalities with a tolerance of ±1 UAH. Dropping observations that fail any check.

In [12]:
TOLERANCE = 1.0

def additive_check(df, col_a, col_b, col_total):
    """Return True where col_a + col_b ≈ col_total (within tolerance), or any value is NaN."""
    return (
        (df[col_a] + df[col_b] - df[col_total]).abs() <= TOLERANCE
    ) | df[[col_a, col_b, col_total]].isna().any(axis=1)

def lte_check(df, col_part, col_total):
    """Return True where col_part <= col_total + tolerance, or any value is NaN."""
    return (
        df[col_part] <= df[col_total] + TOLERANCE
    ) | df[[col_part, col_total]].isna().any(axis=1)


initial_obs = len(df)

checks = {
    'check_assets' : additive_check(df, 'Total Non-Current Assets', 'Total Current Assets', 'Total Assets'),
    'check_liabilities' : additive_check(df, 'Total Long-term Liabilities', 'Total Current Liabilities', 'Total Liabilities'),
    'check_balance' : additive_check(df, 'Total Liabilities', 'Total Equity', 'Total Assets'),
    'check_debt' : additive_check(df, 'Short-Term Bank Loans', 'Long-Term Bank Loans', 'Total Debt'),
    'check_gross_profit' : (
        (df['Net Revenue'] - df['COGS'] - df['Gross Profit']).abs() <= TOLERANCE
    ) | df[['Net Revenue', 'COGS', 'Gross Profit']].isna().any(axis=1),
    'check_ebitda' : additive_check(df, 'EBIT', 'Depreciation', 'EBITDA'),
    'check_ebitda_gte_ebit' : (
        df['EBITDA'] >= df['EBIT'] - TOLERANCE
    ) | df[['EBITDA', 'EBIT']].isna().any(axis=1),
    'check_liab_gte_current' : lte_check(df, 'Total Current Liabilities', 'Total Liabilities'),
    'check_liab_gte_longterm': lte_check(df, 'Total Long-term Liabilities', 'Total Liabilities'),
    'check_debt_gte_short' : lte_check(df, 'Short-Term Bank Loans', 'Total Debt'),
    'check_debt_gte_long' : lte_check(df, 'Long-Term Bank Loans', 'Total Debt'),
}

check_labels = {
    'check_assets' : 'Total Assets = Non-Current + Current Assets',
    'check_liabilities' : 'Total Liabilities = LT + Current Liabilities',
    'check_balance' : 'Total Assets = Total Liabilities + Equity',
    'check_debt' : 'Total Debt = ST + LT Bank Loans',
    'check_gross_profit' : 'Gross Profit = Net Revenue − COGS',
    'check_ebitda' : 'EBITDA = EBIT + Depreciation',
    'check_ebitda_gte_ebit' : 'ERROR if EBITDA < EBIT',
    'check_liab_gte_current' : 'ERROR if Total Liabilities < Current Liabilities',
    'check_liab_gte_longterm' : 'ERROR if Total Liabilities < Long-term Liabilities',
    'check_debt_gte_short' : 'ERROR if Total Debt < Short-Term Bank Loans',
    'check_debt_gte_long' : 'ERROR if Total Debt < Long-Term Bank Loans',
}

print('Balance sheet integrity checks')
for name, series in checks.items():
    n_fail = (~series).sum()
    pct    = n_fail / len(df) * 100
    status = 'ok ' if n_fail == 0 else 'not ok '
    print(f'{status} {check_labels[name]}')
    if n_fail > 0:
        n_firms = df[~series]['Firm ID'].nunique()
        print(f'   Failed: {n_fail:,} obs ({pct:.2f}%) — {n_firms:,} firms')

# Drop observations that fail any check
all_pass = pd.concat(checks, axis=1).all(axis=1)
df = df[all_pass].reset_index(drop=True)


print(f'Observations removed  : {initial_obs - len(df):,}')
print(f'Observations remaining: {len(df):,}')
print(f"Unique firms remaining: {df['Firm ID'].nunique():,}")

Balance sheet integrity checks
ok  Total Assets = Non-Current + Current Assets
ok  Total Liabilities = LT + Current Liabilities
ok  Total Assets = Total Liabilities + Equity
ok  Total Debt = ST + LT Bank Loans
not ok  Gross Profit = Net Revenue − COGS
   Failed: 131 obs (0.16%) — 62 firms
ok  EBITDA = EBIT + Depreciation
ok  ERROR if EBITDA < EBIT
ok  ERROR if Total Liabilities < Current Liabilities
ok  ERROR if Total Liabilities < Long-term Liabilities
ok  ERROR if Total Debt < Short-Term Bank Loans
ok  ERROR if Total Debt < Long-Term Bank Loans
Observations removed  : 131
Observations remaining: 84,051
Unique firms remaining: 18,695


## 8. Requiring minimum observations per firm

Dropping firms with fewer than two yearly observations, as at least one predictor year (t−1) and one outcome year (t) are needed.

In [13]:
initial_obs   = len(df)
initial_firms = df['Firm ID'].nunique()

obs_per_firm  = obs_count_per_firm(df)
firms_to_keep = obs_per_firm[obs_per_firm['Num Years'] >= 2]['Firm ID']
df = df[df['Firm ID'].isin(firms_to_keep)].reset_index(drop=True)

print('Firms with fewer than 2 observations — removed')
print(f'Firms removed         : {initial_firms - df["Firm ID"].nunique():,}')
print(f'Observations removed  : {initial_obs - len(df):,}')
print(f'Firms remaining       : {df["Firm ID"].nunique():,}')
print(f'Observations remaining: {len(df):,}')

Firms with fewer than 2 observations — removed
Firms removed         : 157
Observations removed  : 157
Firms remaining       : 18,538
Observations remaining: 83,894


## 9. Requiring consecutive year pairs

Filtering out observations that do not belong to any consecutive (t−1, t) pair within the same firm. Each outcome year t requires financial data at t−1 to construct lagged predictors.

In [14]:
initial_obs   = len(df)
initial_firms = df['Firm ID'].nunique()

df = df.sort_values(['Firm ID', 'Year']).reset_index(drop=True)

# Identify consecutive year pairs
df['_year_prev'] = df.groupby('Firm ID')['Year'].shift(1)
df['_year_next'] = df.groupby('Firm ID')['Year'].shift(-1)

has_prev = df['Year'] - df['_year_prev'] == 1  # this row is a valid t
has_next = df['_year_next'] - df['Year'] == 1  # this row is a valid t-1

df = df[has_prev | has_next].reset_index(drop=True)
df = df.drop(columns=['_year_prev', '_year_next'])

print('Removed observations without consecutive year pairs')
print(f'Observations removed  : {initial_obs - len(df):,}')
print(f'Firms removed         : {initial_firms - df["Firm ID"].nunique():,}')
print(f'Observations remaining: {len(df):,}')
print(f'Firms remaining       : {df["Firm ID"].nunique():,}')
print('\nObservations per year after filtering:')
print(df.groupby('Year')['Firm ID'].count().astype(int))

Removed observations without consecutive year pairs
Observations removed  : 2,832
Firms removed         : 419
Observations remaining: 81,062
Firms remaining       : 18,119

Observations per year after filtering:
Year
2018.0    10888
2019.0    12493
2020.0    10651
2021.0    12603
2022.0    12086
2023.0    11867
2024.0    10474
Name: Firm ID, dtype: int64


In [15]:
df.head(10)

,Firm ID,Firm Status,Year,KVED,Sector,Date Registration,Firm Age,Region,Number of Employees,Intangible Assets,...,Employee Benefits Expense (Wages and Salaries),Social Security Contributions,Depreciation,Other Operating Expenses (by nature),Total Operating Expenses,Weighted Average Number of Ordinary Shares,Diluted Weighted Average Number of Ordinary Shares,Basic Earnings (Loss) per Share,Diluted Earnings (Loss) per Share,Dividends per Ordinary Share
0,1,Припинено,2018.0,20.14,C,08.12.2003,16.0,Чернівецька область,0.0,0.0,...,154.0,26.0,105.0,456.0,748.0,0.0,0.0,0.0,0.0,0.0
1,1,Припинено,2019.0,20.14,C,08.12.2003,17.0,Чернівецька область,0.0,0.0,...,195.0,35.0,105.0,228.0,571.0,0.0,0.0,0.0,0.0,0.0
2,1,Припинено,2021.0,20.14,C,08.12.2003,19.0,Чернівецька область,0.0,0.0,...,410.0,72.0,90.0,947.0,1529.0,0.0,0.0,0.0,0.0,0.0
3,1,Припинено,2022.0,20.14,C,08.12.2003,20.0,Чернівецька область,0.0,0.0,...,405.0,87.0,87.0,1874.0,2456.0,0.0,0.0,0.0,0.0,0.0
4,1,Припинено,2023.0,20.14,C,08.12.2003,21.0,Чернівецька область,0.0,0.0,...,419.0,92.0,34.0,3882.0,4432.0,0.0,0.0,0.0,0.0,0.0
5,2,Не перебуває в процесі припинення,2023.0,01.46,A,06.09.1994,30.0,Львівська область,214.0,0.0,...,15643.0,3472.0,25568.0,65079.0,415358.0,0.0,0.0,0.0,0.0,0.0
6,2,Не перебуває в процесі припинення,2024.0,01.46,A,06.09.1994,31.0,Львівська область,214.0,0.0,...,39686.0,8707.0,37187.0,142440.0,1123592.0,0.0,0.0,0.0,0.0,0.0
7,3,В стані припинення,2020.0,02.40,A,29.03.1999,22.0,Харківська область,0.0,26.0,...,23982.0,5315.0,2403.0,2337.0,59121.0,0.0,0.0,0.0,0.0,0.0
8,3,В стані припинення,2021.0,02.40,A,29.03.1999,23.0,Харківська область,0.0,3.0,...,19529.0,4242.0,2319.0,7833.0,62279.0,0.0,0.0,0.0,0.0,0.0
9,3,В стані припинення,2022.0,02.40,A,29.03.1999,24.0,Харківська область,0.0,0.0,...,22574.0,4763.0,2239.0,7334.0,63495.0,0.0,0.0,0.0,0.0,0.0


## 7. Saving cleaned dataset

Writing the cleaned DataFrame to CSV for use in subsequent notebooks.

In [16]:
OUTPUT_PATH = 'df_cleaned.csv'

df.to_csv(OUTPUT_PATH, sep=';', index=False, encoding='utf-8')

print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: {df.shape}')

Saved: df_cleaned.csv
Shape: (81062, 167)
